# Music Generation I

## Exercise 1 [Markov Models, 4 points]

In [378]:
import os
import pathlib
import random
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Protocol, Tuple, Dict

import math
import music21
import torch
import torch.nn.utils as F
from music21 import converter
from pomegranate.markov_chain import MarkovChain

In [367]:
current_dir = pathlib.Path.cwd()
data_path = current_dir.joinpath("data")
output_path = data_path.joinpath("output")

# File extensions
ABC_FILE_EXTENSION = ".abc"
MUSIC_XML_FILE_EXTENSION = ".xml"
MIDI_FILE_EXTENSION = ".midi"

# Data
STYLES = ["french", "irish_folk"]
ORDERS = [1, 2, 4]

# Constants
N_BARS = 16

In [368]:

@dataclass(frozen=True)
class State:
    note: int
    duration: int

    #score_position: Optional[int] = None
    #bar_position: Optional[int] = None
    #articulation: Optional[int] = None
    # add fields as you discover them

    def to_music21(self):
        if self.note == 0:
            return music21.note.Rest(quarterLength=self.duration)
        else:
            return music21.note.Note(self.note, quarterLength=self.duration)


class Embedding(Protocol):
    def initialize_embedding(self, states: list[State]) -> None:
        """ Initializes embedding. """

    def encode(self, state: State) -> int:
        """Project a State to a discrete token."""

    def decode(self, idx: int) -> State:
        """Optional: reconstruct a State (partial is acceptable)."""

    def get_eos_idx(self) -> int:
        """Index used for end of sentence and padding."""

    def __len__(self) -> int:
        """Vocabulary size."""


class DistinctStateEmbedding(Embedding):
    """
    One embedding for each distinct symbol.
    """

    def __init__(self):
        self.state2idx = None
        self.idx2state = None
        self.eos_idx = 0

    def initialize_embedding(self, states: list[State]):
        unique_states = list(set(states))
        self.state2idx = {s: i + 1 for i, s in enumerate(unique_states)}
        self.idx2state = {i + 1: s for i, s in enumerate(unique_states)}

    def encode(self, state: State) -> int:
        return self.state2idx[state]

    def decode(self, idx: int) -> Optional[State]:
        if idx == self.eos_idx:
            return None
        assert idx <= len(self.idx2state), f"idx out of range (idx {idx} > {len(self.idx2state)} vocab_size)"
        return self.idx2state[idx]

    def get_eos_idx(self) -> int:
        return self.eos_idx

    def __len__(self) -> int:
        return len(self.state2idx)

In [369]:
def get_state_seq(file: pathlib.Path) -> list[State]:
    """
    For now only storing pitch and duration of the notes and rests
    :param file:
    :return:
    """
    states = []

    score = converter.parse(file)
    for el in score.recurse():  # TODO: this logic could be in the state...
        if isinstance(el, music21.note.Note):
            states.append(State(note=el.pitch.midi, duration=el.duration.quarterLength))
        elif isinstance(el, music21.note.Rest):
            states.append(State(note=0, duration=el.duration.quarterLength))
        else:
            continue

    return states


def get_samples(style_dir: pathlib.Path, embedding_class: type[Embedding]) -> Tuple[torch.Tensor, Embedding]:
    # Get state sequences
    state_sequences: list[list[State]] = []
    all_states: list[State] = []
    for file in sorted(style_dir.glob(f"*{ABC_FILE_EXTENSION}")):
        state_seq = get_state_seq(file)
        state_sequences.append(state_seq)
        all_states.extend(state_seq)

    # Initialize embedding
    embedding = embedding_class()
    embedding.initialize_embedding(all_states)

    # Get tensor
    tensor_list = [torch.tensor([embedding.encode(el) for el in seq]) for seq in state_sequences]
    X = F.rnn.pad_sequence(tensor_list, batch_first=True, padding_value=embedding.get_eos_idx())
    X = X.unsqueeze(-1)
    return X, embedding


def create_and_fit_markov_model(data: torch.Tensor, order) -> MarkovChain:
    model = MarkovChain(k=order)
    model.fit(data)
    return model


def sample_from_probs(probs: torch.Tensor, generator: torch.Generator, eos_idx: int) -> int:
    """
    probs: 1D tensor summing to 1
    """
    probs_wo_eos = probs.clone()
    probs_wo_eos[eos_idx] = 0.0
    total_mass = probs_wo_eos.sum()
    if total_mass == 0:
        return eos_idx
    else:
        probs_wo_eos /= total_mass
        return torch.multinomial(probs_wo_eos, num_samples=1, generator=generator).item()


def get_random_initial_state(n_samples: int, vocab_size: int, seed: int) -> list[int]:
    """
    :param n_samples:
    :param vocab_size:
    :return:
    """
    random.seed(seed)
    return [random.randint(0, vocab_size) for _ in range(n_samples)]


def get_duration_of_idx_seq(seq: list[int], embedding: Embedding) -> float:
    return sum([embedding.decode(i).duration for i in seq])


def generate_seq(model: MarkovChain, max_samples: Optional[int], embedding: Embedding, seed: int, n_bars: int = 4, ) -> \
        list[int]:
    seq = get_random_initial_state(n_samples=model.k, vocab_size=len(embedding), seed=seed)

    if seed is not None:
        gen = torch.Generator()
        gen.manual_seed(seed)
    else:
        gen = None

    target_score_duration = n_bars * 4  #ASSUMING 4/4 and quarternote duration!!    

    i = 0
    score_duration = get_duration_of_idx_seq(seq, embedding)
    while score_duration < target_score_duration:
        dist = model.distributions[model.k].probs[0]
        context = seq[-model.k:]
        probs_i = dist[tuple(context)]
        i_idx = sample_from_probs(probs_i, generator=gen, eos_idx=embedding.get_eos_idx())
        seq.append(i_idx)
        i += 1
        score_duration += embedding.decode(i_idx).duration
        if max_samples is not None and i >= max_samples:
            break

    return seq


# def trim_part_to_n_measures(part: music21.stream.Part, n: int):
#     """
#     Keeps only the first n measures of a music21 Part.
#     Modifies the Part in place.
#     """
#     measures = list(part.measures(0, None))
#
#     if len(measures) <= n:
#         print(f"{len(measures)}, {n}")
#         return  # nothing to trim
#
#     for m in measures[n:]:
#         part.remove(m)
#
#     print(len(part.measures(0, None)))


def export_results(idx_seq: List[int], embedding: Embedding, out_path: pathlib.Path, name: str,
                   n_bars: int,
                   store_xml: bool = True,
                   store_midi: bool = True,
                   store_score_image: bool = False,
                   store_score_audio: bool = False,
                   ) -> music21.stream.Score:
    state_seq = [s for s in [embedding.decode(x) for x in idx_seq] if s is not None]

    stream = music21.stream.Stream()
    for state in state_seq:
        stream.append(state.to_music21())

    score = music21.stream.Score()
    part = music21.stream.Part()
    part.append(stream.makeMeasures())
    # trim_part_to_n_measures(part, n_bars)
    score.append(part)

    # Metadata
    score.insert(0, music21.metadata.Metadata())
    score.metadata.title = name
    score.metadata.composer = "generated using a Markov Chain"

    if out_path is not None:
        # XML
        if store_xml:
            score_tmp_path = pathlib.Path(score.write('musicxml'))
            music_xml_str = score_tmp_path.read_text()
            file_path = out_path.joinpath(f"{name}{MUSIC_XML_FILE_EXTENSION}")
            with open(file_path, "w+") as f:
                f.write(music_xml_str)

        # MIDI
        # if store_midi:
        #     score_tmp_path = pathlib.Path(score.write('midi'))
        #     midi_str = score_tmp_path.read_text()
        #     file_path = out_path.joinpath(f"{name}{MIDI_FILE_EXTENSION}")
        #     with open(file_path, "w+") as f:
        #         f.write(midi_str)

        # SCORE
        if store_score_image:
            pass  # TODO: !!!

        # AUDIO
        if store_score_audio:
            pass  # TODO: !!!

    return score


In [370]:
def get_seqs(styles: List[str],
             orders: List[int],
             embedding_class: Optional[type[Embedding]] = DistinctStateEmbedding,
             verbose: bool = False,
             max_samples: int = 250,
             n_bars: int = 16,
             seed: Optional[int] = 42,
             save_files: bool = True) -> Tuple[pathlib.Path, Dict[str, MarkovChain], Dict[str, music21.stream.Score]]:
    # OUT PATH
    if save_files:
        out_folder_name = f"{datetime.now().strftime('%y%m%d-%H%M%S')}"
        out_folder_path = output_path.joinpath(out_folder_name)
        if not os.path.exists(out_folder_path):
            os.mkdir(out_folder_path)
    else:
        out_folder_path = None

    models = {}
    scores = {}
    for style in styles:
        # DATA
        if verbose:
            print(f"\nProcessing -> {style}")
        data, embedding = get_samples(data_path.joinpath(style), embedding_class)
        if verbose:
            print(f"\tVocabulary size: {len(embedding)}")
            #print(f"\tdata shape: {data.shape}")

        for order in orders:
            model_name = get_model_name(order, style)

            # TRAINING
            if verbose:
                print(f"\n\tTraining model with order {order}", end="")
            model = create_and_fit_markov_model(data=data, order=order)
            models[model_name] = model
            if verbose:
                print("\t -> Done!")

            # INFERENCE
            idx_seq = generate_seq(model, max_samples=max_samples, n_bars=n_bars, embedding=embedding, seed=seed)
            #if verbose:
            #    print(f"\t{idx_seq}")

            # EXPORT
            score = export_results(idx_seq, embedding=embedding, n_bars=n_bars, out_path=out_folder_path,
                                   name=model_name)
            scores[model_name] = score

    return out_folder_path, models, scores


def get_model_name(order: int, style: str) -> str:
    return f"{style}-{order}"


In [371]:
_, models, scores = get_seqs(STYLES, ORDERS, n_bars=16, verbose=True)

# Demo
#_, models, _ = get_seqs(["french"], orders=[1], n_bars=16, verbose=True, seed=42, save_files=True)


Processing -> french
	Vocabulary size: 40

	Training model with order 1	 -> Done!

	Training model with order 2	 -> Done!

	Training model with order 4	 -> Done!

Processing -> irish_folk
	Vocabulary size: 31

	Training model with order 1	 -> Done!

	Training model with order 2	 -> Done!

	Training model with order 4	 -> Done!


In [372]:
## Plot distributions

# probs_np = probs.detach().cpu().numpy()
# plt.figure()
# plt.bar(range(len(probs_np)), probs_np)
# plt.xlabel("Index / Class")
# plt.ylabel("Probability")
# plt.title("Probability Distribution")
# plt.show()


## Exercise 2 [Evaluation, 2 points]

In [375]:
class EvaluationMethod(Protocol):

    @staticmethod
    def evaluate(score: music21.stream.Score) -> float:
        """Returns a scalar given a piece."""

In [387]:
class MelodicSmoothness(Protocol):
    """
    Penalizes large intervals.
    Calculated averaging average pitch distance between intervals and exp.
    To maximize.
    """

    @staticmethod
    def evaluate(score: music21.stream.Score, sigma: float = 3.5, var: float = 3) -> float:
        """
        :param score:
        :param tau: with 3.0 you add tolerance to up to avg distance of 6. More than that seems too much
        :return:
        """
        notes = list(score.recurse().notes)
        prev_pitch = notes[0].pitch.midi
        res = 0
        for n in notes:
            if isinstance(n, music21.note.Note):
                res += abs(prev_pitch - n.pitch.midi)
                prev_pitch = n.pitch.midi

        abs_mean = res / (len(notes) - 1)
        return math.exp(-(abs_mean - sigma) ** 2 / (2 * var ** 2))


print(f"Using evaluation method {EvaluationMethod.__name__}")
for score_name, score in scores.items():
    score = MelodicSmoothness.evaluate(score)
    print(f"\t{score_name} -> {round(score, 5)}")

Using evaluation method EvaluationMethod
	french-1 -> 0.81857
	french-2 -> 0.96663
	french-4 -> 1.0
	irish_folk-1 -> 0.96777
	irish_folk-2 -> 0.93371
	irish_folk-4 -> 0.56903


In [ ]:
#TODO: add some rythm metric

In [ ]:
#TODO: add some harmoic metric

In [ ]:
#TODO: combine them into a metric of goodness (normalize all between 0 and 1 or so...

## Exercise 3 [Evolutionary Algorithms, 4 points]

In [ ]:
# TODO:!!